# Stage 2 — Synthetic Data Generation

**Project:** LLM-Based Information Extraction from Hospitality Request Emails: A Prompt Engineering Approach

**Author:** Mikail Imran Shaukat

---

## What this notebook does

This notebook generates the synthetic email dataset used for extraction and evaluation in Stages 3 and 4.

The 17 real emails collected from Xebia's hospitality inbox (Stage 1) were not sufficient to produce statistically meaningful field-level precision, recall, and F1 scores across the 12-field extraction schema. Synthetic data generation was therefore used to create a dataset of 50 emails.

**Input:** `Stage 2 - Extraction Schema & Hospitality Emails.xlsx` = The 17 real emails from Stage 1  
**Output:** `Stage 2 - Synthetic Data.csv` = 50 synthetic emails ready for extraction and evaluation

---
## Cell 1 — Setup

Install and import the required libraries, and initialise the OpenAI client.

The API key is stored securely in Colab secrets under the name `OPENAI_API_KEY` and is never hardcoded in the notebook.

In [2]:
!pip install openai -q

import json
import time
import pandas as pd
from google.colab import userdata
from openai import OpenAI

client = OpenAI(api_key=userdata.get('OPENAI_API_KEY'))
print('Client set up successfully')

Client set up successfully


---
## Cell 2 — Load Real Emails

The 17 real emails from Stage 1 are loaded here and used directly as stylistic references in the generation prompt.

Including the real emails in the prompt rather than describing their style in abstract terms is a deliberate design choice. Including real-world examples as references produces synthetic outputs that more closely reflect the structure and variety of the real data compared to generation without examples. The real emails carry all the stylistic weight and guide the model in generation.

In [3]:
df_real = pd.read_excel('Stage 1 - Extraction Schema & Hospitality Emails.xlsx', sheet_name='Real Emails', header = 3)
real_emails = df_real['Email Body (Translated)'].dropna().tolist()

print(f'Loaded {len(real_emails)} real emails')

Loaded 17 real emails


---
## Cell 3 — Build Generation Prompt

The generation prompt is built here as a function so it can be called fresh for each batch.

**Key design decisions in the prompt:**

**1. Real emails as the only stylistic reference**  
All 17 real emails are embedded directly in the prompt. The model is instructed to treat these as its only reference for how emails are written and not as a checklist of what to include. Earlier prompt versions included explicit tone and style rules (e.g. *"write informally"*, *"vary the length"*) but these produced emails that were too clean and corporate. Removing these rules and letting the real examples carry the style produced much more realistic outputs in line with the real emails.

**2. Field frequency guidance**  
The 12 extraction schema fields are listed with their occurrence rates calculated from the real email annotations in Stage 1. These percentages guide how often each field should appear across the 50 generated emails, ensuring the distribution of the synthetic dataset mirrors and real data set matches. Fields with zero occurrence in the real data (av_equipment) are explicitly instructed to appear in at least a small number of emails to ensure edge cases are covered.

**3. Structured JSON output**  
The prompt requests a structured JSON array to make the output directly parsable.

In [4]:
examples_text = ''
for i, email in enumerate(real_emails):
    examples_text += f'Email {i+1}:\n{email}\n\n'

def build_prompt(batch_size):
    return f"""You are generating synthetic hospitality request emails for Xebia, a technology
consultancy in the Netherlands. Employees write emails to Xebia's hospitality
staff in order to organise a wide range of events.

Below are 17 real emails from Xebia's hospitality inbox. These are your only
reference for how these emails are written.

{examples_text}

Generate {batch_size} emails that could have come from this same inbox. The emails
should read like the sender is thinking out loud. Some ask questions mid-email,
some remember extra details halfway through, some are apologetic about late notice,
some are vague because the sender does not have all the information yet.

Important: The field list below is for frequency guidance only. It is not a
checklist. Do not write emails that systematically include or exclude fields.
Write naturally and inconsistently, just as the real examples do.

- event_date: The date the event needs to take place. e.g. 12 May (100%)
- event_type: The purpose of the event — always implied, never stated explicitly. e.g. Partner Meeting (100%)
- meal_type: The type of meal to be included, if any. e.g. Lunch (88%)
- attendee_count: Number of people expected to attend. e.g. 20 (71%)
- external_guests: Whether the event includes external guests and who they are. e.g. Microsoft (47%)
- room_location: The specific room or area within the office. e.g. Grand Cafe (82%)
- event_time: Start and end time of the event. e.g. 10:00 to 14:00 (65%)
- catering_type: The type of catering required given that a meal is included. e.g. Normal Buffet (41%)
- office_location: The Xebia office at which the event will take place. e.g. Amsterdam (24%)
- extra_arrangements: Any special or additional arrangements. e.g. Platter with goodies (29%)
- dietary_requirements: Any dietary restrictions among attendees. e.g. Vegetarian (24%)
- av_equipment: Whether audio-visual equipment is required and what. e.g. Projector (0% in real emails, make them appear naturally in a small number of synthetic emails)

Return ONLY a valid JSON array where each element has exactly two keys:
- \"email_id\": a placeholder number starting from 1
- \"email_text\": the full email text as a string

No extra text, no markdown, no code blocks. Just the raw JSON array."""

print('Prompt function ready')

Prompt function ready


---
## Cell 4 — Generate Emails in Batches and Save

This cell runs the generation loop and saves the output to `Stage 2 - Synthetic Data.csv`.

**Why batches of 5?**  
Initial attempts generated all 50 emails in a single API call. This consistently produced structurally repetitive outputs, as if the model was following a template. To solve this, generation was done in batches of 5 emails 10 API calls. Each batch independently processes the 17 real emails and produces more varied outputs.

**JSON cleaning**  
Despite explicit instructions to return raw JSON only, the OpenAI API occasionally wraps output in markdown code fences (` ```json ``` `). A cleaning step is applied to each batch response before parsing to strip any such fences, preventing `json.loads()` from failing.

**Temperature**  
Set to 0.8 to encourage stylistic variation across generated emails. In line with the literature reviewed within the thesis writing.

In [ ]:
all_emails = []
batch_size = 5
num_batches = 10

for batch in range(num_batches):
    print(f'Generating batch {batch + 1}/{num_batches}...', end=' ')

    response = client.chat.completions.create(
        model='gpt-4o',
        messages=[{'role': 'user', 'content': build_prompt(batch_size)}],
        temperature=0.8,
    )

    raw_output = response.choices[0].message.content.strip()
    raw_output = raw_output.strip().removeprefix('```json').removeprefix('```').removesuffix('```').strip()
    batch_emails = json.loads(raw_output)

    all_emails.extend(batch_emails)
    print(f'Done — {len(batch_emails)} emails generated')

# Making sure email_ids are assigned correctly for the csv file
for i, email in enumerate(all_emails):
    email['email_id'] = i + 1

df_synthetic = pd.DataFrame(all_emails)
df_synthetic.to_csv('Stage 2 - Synthetic Data.csv', index=False)

print(f'\nDone. {len(all_emails)} emails saved to Stage 2 - Synthetic Data.csv')

Generating batch 1/10... Done — 5 emails generated
Generating batch 2/10... 